# Is your backtest real? Check it in 30 seconds

[TokIO](https://github.com/Kuzi-DEV/Tokio-ai) asks whether a backtest's profit is distinguishable from luck, once two things are allowed for:

1. **Serial dependence.** Positions held for weeks make daily P&L far from independent; a plain t-test counts every bar as fresh evidence.
2. **Your search.** The Sharpe on your screen is the best of every variant you tried. The best of twenty zero-edge strategies looks good by construction.

Run the cells top to bottom. No API key, no account. The outputs saved here are from a run on 2026-10-09; yours will move a little as new days come in.

In [1]:
%pip install -q "tokio-ai>=0.6" yfinance

## 1. A grid everyone has run: SMA crossovers on SPY

Nineteen fast/slow pairs, long when fast > slow and short otherwise, on 20 years of daily SPY. The position is decided at the close and held over the next bar, so there is no lookahead.

In [2]:
import pandas as pd
import yfinance as yf
import tokio_ai

close = yf.download("SPY", period="20y", auto_adjust=True, progress=False)["Close"].squeeze()
r = close.pct_change()

pnl, positions = {}, {}
for fast in (5, 10, 20, 50):
    for slow in (50, 100, 150, 200, 250):
        if fast >= slow:
            continue
        f, s = close.rolling(fast).mean(), close.rolling(slow).mean()
        pos = (f > s).astype(float) * 2 - 1          # +1 long, -1 short
        pos = pos.where(s.notna()).shift(1)           # known at the close, held over the next bar
        name = f"sma{fast}/{slow}"
        pnl[name], positions[name] = pos * r, pos
print(len(pnl), "variants")

19 variants


## 2. The best one, the way it usually gets reported

In [3]:
best = max(pnl, key=lambda k: pnl[k].mean() / pnl[k].std())
print(best)
print(tokio_ai.check_backtest(pnl[best], positions=positions[best], asset_returns=r))

sma50/250


SIGNIFICANT (p=0.0394, alpha=0.05).
Sharpe 0.36 annualized over 4781 bars (~19.0 years at 252/year). A plain t-test would say p=0.0588; allowing for serial dependence in the P&L, p=0.0394 before any correction for trials.
Probabilistic Sharpe Ratio: 0.96 allowing for serial dependence; 0.94 as published, which treats bars as independent (0.95 passes).
Minimum track record length at 95%: 4020 bars (have 4781).
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

No costs added here (returns are taken as already net of any fees they include); the strategy turns over 2.0x a year.
Breakeven: it stays significant up to 17.9 bps per unit traded. If your real costs, slippage included, are higher, it's not evidence.
Without its best and worst 48 bars, the Sharpe is 0.50 (from 0.36): still positive.
Made money in 6 of 8 equal time blocks.
Rebuilt from its positions (no costs) the Sharpe is 0.36; with every position taken one bar later, 0.35.

## 3. The same backtest, honestly: every variant you tried

Pass the whole grid. The verdict is about the best variant, corrected for all of them using how correlated they are. You also get the Deflated Sharpe Ratio, the probability of backtest overfitting (does the in-sample winner keep winning out of sample?) and robustness checks.

In [4]:
print(tokio_ai.check_backtest(pnl, positions=positions, asset_returns=r))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1235, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.35 annualized over 4781 bars (~19.0 years at 252/year). A plain t-test would say p=0.0620; allowing for serial dependence in the P&L, p=0.0217 before any correction for trials.
Haircut Sharpe after 19 trials: 0.20.
Deflated Sharpe Ratio of sma50/250 (the DSR tests the highest Sharpe; the verdict's variant is the one closest to passing once dependence is allowed for): 0.79 allowing for serial dependence; 0.76 as published, which treats bars as independent (0.95 passes).
It would pass as the best of at most 2 independent trials. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.35    0.0620    0.0217    0.1235     0.20  not significant
sma5/100      0.32    0.0835    0.0345    0.1787     0.16  not significant
sma10/200     0.35    0.0637    0.0385    0.1908    

## 4. With trading costs

`costs=0.0005` charges 5 bps per unit of position traded. The output also gives the **breakeven cost**: the most you could pay and still have a significant result.

In [5]:
print(tokio_ai.check_backtest(pnl, positions=positions, asset_returns=r, costs=0.0005))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1514, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.34 annualized over 4781 bars (~19.0 years at 252/year). A plain t-test would say p=0.0707; allowing for serial dependence in the P&L, p=0.0277 before any correction for trials.
Haircut Sharpe after 19 trials: 0.18.
Deflated Sharpe Ratio of sma50/250 (the DSR tests the highest Sharpe; the verdict's variant is the one closest to passing once dependence is allowed for): 0.76 allowing for serial dependence; 0.73 as published, which treats bars as independent (0.95 passes).
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.34    0.0707    0.0277    0.1514     0.18  not significant
sma50/250     0.35    0.0614    0.0422    0.2107     0.17  not significant
sma10/200     0.34    0.0685    0.0434    0.2121     

## 5. A backtest that cheats

This one goes long on days that close up, deciding at that day's close: it uses information it couldn't have had. Every significance test passes it. The one-bar delay check doesn't.

In [6]:
cheat = (r > 0).astype(float) * 2 - 1      # knows today's return when it takes today's position
print(tokio_ai.check_backtest((cheat * r).dropna(), positions=cheat[r.notna()], asset_returns=r.dropna()))

SIGNIFICANT (p=0.0000, alpha=0.05).
Sharpe 13.26 annualized over 5030 bars (~20.0 years at 252/year). A plain t-test would say p=0.0000; allowing for serial dependence in the P&L, p=0.0000 before any correction for trials.
Probabilistic Sharpe Ratio: 1.00 allowing for serial dependence; 1.00 as published, which treats bars as independent (0.95 passes).
Minimum track record length at 95%: 77 bars (have 5030).
It would pass as the best of over a million independent trials.

No costs added here (returns are taken as already net of any fees they include); the strategy turns over 254.1x a year.
Breakeven: it stays significant up to 73.4 bps per unit traded. If your real costs, slippage included, are higher, it's not evidence.
Without its best and worst 50 bars, the Sharpe is 16.14 (from 13.26): still positive.
Made money in 8 of 8 equal time blocks.
Rebuilt from its positions (no costs) the Sharpe is 13.26; with every position taken one bar later, -0.44. Most of the edge needs same-bar exec

## 6. No backtest yet? Let TokIO run it

`tokio_ai.Lab` backtests for you, and is built so the usual mistakes can't happen quietly:

* **Lookahead is tested.** The lab re-runs your signal with the future cut off; if any past position changes, it refuses.
* **Realistic fills.** Decided at the close, filled at the next open, 10 bps of costs.
* **Every variant is counted**, so the verdict is corrected for everything you tried.
* **The last 25% of the data is locked away** until one final test.

`vs="market"` asks whether the strategy beat simply holding SPY at the same exposure, so a long-only rule can't pass just because the market rose.

In [7]:
lab = tokio_ai.Lab("SPY", vs="market")           # full daily history, dividend-adjusted

def sma_cross(d, fast=20, slow=100):              # d = the bars so far; return a position per bar
    f, s = d.close.rolling(fast).mean(), d.close.rolling(slow).mean()
    return (f > s).astype(float)                  # 1 = long, 0 = cash

lab.sweep(sma_cross, fast=[10, 20, 50], slow=[100, 150, 200])
print(lab.check())

NOT SIGNIFICANT after correcting for 9 trials (Romano-Wolf, using their correlation) (p=0.3007, alpha=0.05). Strongest variant: sma_cross(fast=50, slow=200).
Sharpe 0.19 annualized over 6361 bars (~25.3 years at 251.742/year). A plain t-test would say p=0.1686; allowing for serial dependence in the P&L, p=0.1327 before any correction for trials.
Haircut Sharpe after 9 trials: 0.09.
Deflated Sharpe Ratio of sma_cross(fast=50, slow=200): 0.57 allowing for serial dependence; 0.56 as published, which treats bars as independent (0.95 passes).
It would not pass even as the only strategy ever tried.

variant                        Sharpe   p naive   p alone     p adj  haircut  verdict
sma_cross(fast=50, slow=200)     0.19    0.1686    0.1327    0.3007     0.09  not significant
sma_cross(fast=20, slow=200)     0.15    0.2210    0.1673    0.3512     0.06  not significant
sma_cross(fast=10, slow=200)     0.15    0.2317    0.1856    0.3738     0.05  not significant
sma_cross(fast=50, slow=150)   

A signal that peeks at tomorrow's close is refused, with the exact bar:

In [8]:
def peeks(d):
    return (d.close.shift(-1) > d.close).astype(float)   # uses the next bar: lookahead

try:
    lab.run(peeks)
except tokio_ai.LookaheadError as e:
    print(e)

peeks() uses future data: its position at 2009-10-14 00:00:00 was 1 on the full data but 0 when the data stops at 2009-10-14 00:00:00. A signal may only use bars up to the one it's decided on (look for shift(-k), center=True, or statistics over the whole series).


Pick your favourite variant and spend the holdout on it: the one honest look at data it has never seen. It works once.

In [9]:
print(lab.final_test(lab.best()))

sma_cross(fast=50, slow=200) on holdout data: 2120 bars, 8 position changes.
NOT SIGNIFICANT (p=0.7576, alpha=0.05).
Sharpe -0.28 annualized over 2120 bars (~8.4 years at 251.742/year). A plain t-test would say p=0.7907; allowing for serial dependence in the P&L, p=0.7576 before any correction for trials.
Probabilistic Sharpe Ratio: 0.20 allowing for serial dependence; 0.21 as published, which treats bars as independent (0.95 passes).
It would not pass even as the only strategy ever tried.

No costs added here (returns are taken as already net of any fees they include); the strategy turns over 1.1x a year.
Without its best and worst 21 bars, the Sharpe is -0.36 (from -0.28).
Made money in 4 of 8 equal time blocks. About what a coin flip gives.
Lab: fills at the next bar's open; 10.0 bps per unit traded (commission + slippage) is already charged in these returns; 1 variant(s) counted.
Tested against the market: each variant's returns minus holding the asset at that variant's average exp

Prefer to just describe it? `pip install "tokio-ai[agent]"` and ask the agent in plain English ("buy when the 20-day average crosses above the 100-day..."): it writes the signal, runs it through this same lab, and explains the verdict.

## 7. Your own backtest

**From vectorbt or backtesting.py**, pass the object itself; returns and positions are read from it:

```python
pf = vbt.Portfolio.from_signals(close, entries, exits, fees=0.001)   # a whole grid works too
print(tokio_ai.check_backtest(pf))

stats = Backtest(data, MyStrategy, commission=0.001).run()
print(tokio_ai.check_backtest(stats))
```

**From TradingView**, download the Strategy Tester's "List of trades" and run:

```python
print(tokio_ai.check_backtest(tokio_ai.read_tradingview("List of trades.csv"), trials=30))
```

**From anything else**, upload a CSV with one column of per-bar returns per variant (a date column is skipped), then run the cell below. If you tried more variants than you're uploading, set `TRIALS` to the real number.

In [10]:
TRIALS = None   # e.g. 40 if you tried 40 things and kept these

try:
    from google.colab import files
    name = next(iter(files.upload()))
except ImportError:
    name = input("path to your CSV: ")
header = open(name, encoding="utf-8-sig").readline()
if "Trade #" in header:                      # a TradingView export
    print(tokio_ai.check_backtest(tokio_ai.read_tradingview(name), trials=TRIALS))
else:
    df = pd.read_csv(name).select_dtypes("number")
    print(tokio_ai.check_backtest(df if df.shape[1] > 1 else df.iloc[:, 0], trials=TRIALS))

Found something odd? [Open an issue](https://github.com/Kuzi-DEV/Tokio-ai/issues/new/choose); there's a template for statistical concerns. The false-positive rates behind every number here are measured and reproducible: [docs/calibration.md](https://github.com/Kuzi-DEV/Tokio-ai/blob/main/docs/calibration.md).